# Med-DDPM Training — Conditional MRI Synthesis

Train a **Denoising Diffusion Probabilistic Model (DDPM)** to synthesize brain MRI slices from tumor segmentation masks.

This notebook is designed for **Google Colab** with GPU acceleration.

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone the repository (skip if already mounted)
import os

REPO_URL = "https://github.com/AmineAitLaamim/Mask-to-MRI.git"
PROJECT_DIR = "/content/Mask-to-MRI"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL}
    print("✅ Repository cloned.")
else:
    print("✅ Repository already exists. Pulling latest changes...")
    %cd $PROJECT_DIR
    !git pull
    print("\n⚠️  If you get import errors after a pull, restart the runtime:")
    print("   Runtime → Restart runtime (in the menu above)")
    print("   Then re-run all cells from the top.")

In [ ]:
# Install dependencies
%cd $PROJECT_DIR
!pip install -q torch torchvision albumentations tifffile scikit-image pyyaml tqdm matplotlib

In [ ]:
# Mount Google Drive (optional — for persistent checkpoints)
from google.colab import drive

USE_DRIVE = True  # Set to False to skip Drive mounting

if USE_DRIVE:
    drive.mount('/content/drive')
    DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Mask-to-MRI"
    os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
    # Symlink outputs to Drive for persistence
    if not os.path.exists(os.path.join(PROJECT_DIR, "outputs/checkpoints")):
        os.makedirs(os.path.join(PROJECT_DIR, "outputs"), exist_ok=True)
        os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "checkpoints"), exist_ok=True)
        os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "samples"), exist_ok=True)
        os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "metrics"), exist_ok=True)
        print(f"✅ Drive mounted at {DRIVE_PROJECT_DIR}")

## 2. Upload Dataset

In [ ]:
# Option A: Upload from local (uncomment to use)
from google.colab import files

# Upload your dataset zip file
# uploaded = files.upload()
# Then extract it to data/raw/

# Option B: Download from a URL (uncomment to use)
# !wget -q -O data/raw/lgg-dataset.zip <YOUR_DATASET_URL>
# !unzip -q -o data/raw/lgg-dataset.zip -d data/raw/

# Option C: Use existing data in Drive
# !cp -r /content/drive/MyDrive/Mask-to-MRI/data/raw/* data/raw/

print("✅ Dataset ready. Verify the path below:")
print(f"   Raw data dir: {PROJECT_DIR}/data/raw/lgg-mri-segmentation/")

## 3. Import Modules & Load Config

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

# ── Clear any cached src modules from previous runs ──
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

import torch
import yaml
from pathlib import Path

# Import med_ddpm package
from src.med_ddpm import (
    create_unet,
    DDPM,
    DDIMSampler,            # Fast sampling (50–100 steps vs 1000)
    build_ddpm_dataloaders,
    ddpm_train,
    EMA,
    find_latest_ddpm_checkpoint,
    generate_from_masks,
)

# Load config
config_path = Path(PROJECT_DIR) / "config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("✅ Config loaded:")
print(f"   Timesteps:      {config['med_ddpm']['timesteps']}")
print(f"   Epochs:         {config['med_ddpm']['epochs']}")
print(f"   Batch size:     {config['med_ddpm']['batch_size']}")
print(f"   LR:             {config['med_ddpm']['lr']}")
print(f"   EMA decay:      {config['med_ddpm']['ema_decay']}")
print(f"   Schedule:       {config['med_ddpm']['schedule']}")

## 4. Initialize Model & Data Loaders

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

# Fix seeds for reproducibility
import random
import numpy as np
seed = config["data"]["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
print(f"✅ Random seed fixed: {seed}")

In [ ]:
# Build DataLoaders
ddpm_cfg = config["med_ddpm"]
data_cfg = config["data"]

raw_dir = Path(PROJECT_DIR) / data_cfg["raw_dir"]
if not raw_dir.exists():
    raise FileNotFoundError(
        f"Dataset not found at {raw_dir}.\n"
        f"Please upload the LGG dataset to this location first."
    )

loaders = build_ddpm_dataloaders(
    raw_dir=str(raw_dir),
    image_size=data_cfg["image_size"],
    batch_size=ddpm_cfg["batch_size"],
    num_workers=2,  # Colab has limited workers
    seed=data_cfg["seed"],
    balanced=data_cfg["balanced"],
    tumor_ratio=data_cfg["tumor_ratio"],
)

print(f"✅ DataLoaders created:")
print(f"   Train: {len(loaders['train'].dataset)} samples")
print(f"   Val:   {len(loaders['val'].dataset)} samples")
print(f"   Test:  {len(loaders['test'].dataset)} samples")

In [ ]:
# Verify a batch loads correctly
mask_batch, mri_batch = next(iter(loaders["train"]))
print(f"✅ Batch shape: mask={mask_batch.shape}, mri={mri_batch.shape}")
print(f"   Mask range: [{mask_batch.min():.2f}, {mask_batch.max():.2f}]")
print(f"   MRI range:  [{mri_batch.min():.2f}, {mri_batch.max():.2f}]")

In [ ]:
# Create U-Net model
model = create_unet(
    in_channels=config["model"]["output_channels"],    # 3 (MRI channels)
    cond_channels=config["model"]["input_channels"],   # 1 (mask)
    num_filters=ddpm_cfg["num_filters"],
    time_emb_dim=ddpm_cfg["time_emb_dim"],
    norm=ddpm_cfg["norm"],
    attention_resolutions=ddpm_cfg.get("attention_resolutions", [16]),
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model created: {n_params:,} parameters ({n_params/1e6:.2f}M)")

In [ ]:
# Create DDPM wrapper
ddpm = DDPM(
    model=model,
    timesteps=ddpm_cfg["timesteps"],
    beta_start=ddpm_cfg["beta_start"],
    beta_end=ddpm_cfg["beta_end"],
    schedule=ddpm_cfg["schedule"],
    device=device,
).to(device)

print(f"✅ DDPM initialized: {ddpm.timesteps} timesteps, {ddpm_cfg['schedule']} schedule")

## 5. Check for Resumable Checkpoint

In [ ]:
# Check for existing checkpoints
checkpoint_dir = Path(PROJECT_DIR) / config["paths"]["checkpoints"]
latest_ckpt = find_latest_ddpm_checkpoint(str(checkpoint_dir))

if latest_ckpt:
    print(f"✅ Found checkpoint to resume: {latest_ckpt}")
else:
    print("⚠️  No checkpoint found. Starting from scratch.")

resume_from = str(latest_ckpt) if latest_ckpt else None

## 6. Start Training

In [ ]:
# Run training
history = ddpm_train(
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    model=model,
    ddpm=ddpm,
    config=config,
    device=device,
    checkpoint_dir=str(checkpoint_dir),
    samples_dir=str(Path(PROJECT_DIR) / config["paths"]["samples"]),
    metrics_dir=str(Path(PROJECT_DIR) / config["paths"]["metrics"]),
    resume_from=resume_from,
)

## 7. Plot Training Loss

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
losses = [h["loss"] for h in history]

plt.figure(figsize=(10, 5))
plt.plot(epochs, losses, "b-", linewidth=2)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("MSE Loss", fontsize=12)
plt.title("DDPM Training Loss", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Generate Sample Images

In [ ]:
# Load the best model (EMA weights)
ema = EMA(model)
if latest_ckpt:
    from src.med_ddpm import load_ddpm_checkpoint
    load_ddpm_checkpoint(latest_ckpt, model, ema=ema, device=device)
    ema.apply_shadow()  # Use EMA weights for better quality
    print("✅ Loaded EMA weights for sampling")

model.eval()

In [ ]:
# Generate samples from validation set
import numpy as np
from PIL import Image

mask_batch, real_batch = next(iter(loaders["val"]))
mask_batch = mask_batch.to(device)
real_batch = real_batch.to(device)

# Generate (this takes a while — 1000 timesteps per sample)
with torch.no_grad():
    fake_batch = ddpm.sample(mask_batch[:4])  # Generate 4 samples

# Build comparison grid
rows = []
for i in range(4):
    mask_np = ((mask_batch[i, 0].cpu().numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    real_np = ((real_batch[i].cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    fake_np = ((fake_batch[i].cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    
    mask_3ch = np.stack([mask_np] * 3, axis=-1)
    row = np.concatenate([mask_3ch, fake_np, real_np], axis=1)
    rows.append(row)

grid = np.concatenate(rows, axis=0)

plt.figure(figsize=(15, 15))
plt.imshow(grid)
plt.axis("off")
plt.title("Mask | DDPM Generated | Real", fontsize=16)
plt.tight_layout()
plt.show()

## 9. Save Checkpoint to Drive (Optional)

In [ ]:
if USE_DRIVE:
    import shutil
    
    # Copy checkpoints to Drive
    src_checkpoints = Path(PROJECT_DIR) / "outputs/checkpoints"
    dst_checkpoints = Path(DRIVE_PROJECT_DIR) / "checkpoints"
    
    if src_checkpoints.exists():
        shutil.copytree(src_checkpoints, dst_checkpoints, dirs_exist_ok=True)
        print(f"✅ Checkpoints saved to Drive: {dst_checkpoints}")
    
    # Copy samples to Drive
    src_samples = Path(PROJECT_DIR) / "outputs/samples"
    dst_samples = Path(DRIVE_PROJECT_DIR) / "samples"
    
    if src_samples.exists():
        shutil.copytree(src_samples, dst_samples, dirs_exist_ok=True)
        print(f"✅ Samples saved to Drive: {dst_samples}")

## 10. Evaluate GAN Quality (Optional)

In [ ]:
# Evaluate with SSIM, PSNR
from src.pix2pix import compute_ssim_batch, compute_psnr_batch

ssim = compute_ssim_batch(fake_batch, real_batch[:4])
psnr = compute_psnr_batch(fake_batch, real_batch[:4])

print(f"✅ Evaluation (4 samples):")
print(f"   SSIM: {ssim:.4f}")
print(f"   PSNR: {psnr:.2f} dB")